# Funções de ativação

**Objetivo:** desenhar as ativações e suas derivadas, treinar a mesma rede com sigmoide e com ReLU, e **medir** o gradiente que desaparece nas primeiras camadas de uma rede profunda.

In [ ]:
# bibliotecas base
import numpy as np
import pandas as pd

# Plotly para os gráficos (interativos e leves no Colab)
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.templates.default = "simple_white"

# paleta do curso (a mesma do site)
AZUL, VERMELHO, VERDE = "#3266ad", "#c0392b", "#1a7a4a"
TINTA, SUAVE = "#1c1e15", "#6b7050"

# reprodutibilidade: uma única semente para tudo que é aleatório
SEMENTE = 42
np.random.seed(SEMENTE)
import torch

## 1. As funções e suas derivadas

O que a backpropagation multiplica é a **derivada**. Repare como a da sigmoide e a da tanh somem longe do zero, enquanto a da ReLU é 1 para $z>0$.

In [ ]:
z = np.linspace(-6, 6, 300)
sig = 1/(1+np.exp(-z)); d_sig = sig*(1-sig)
th = np.tanh(z); d_th = 1 - th**2
relu = np.maximum(0, z); d_relu = (z > 0).astype(float)

from plotly.subplots import make_subplots
figura = make_subplots(rows=1, cols=2, subplot_titles=("f(z)", "derivada f'(z)"))
for nome, curva, cor in [("sigmoide", sig, AZUL), ("tanh", th, VERDE), ("ReLU", relu, VERMELHO)]:
    figura.add_trace(go.Scatter(x=z, y=curva, mode="lines", line=dict(color=cor), name=nome), row=1, col=1)
for nome, curva, cor in [("sigmoide", d_sig, AZUL), ("tanh", d_th, VERDE), ("ReLU", d_relu, VERMELHO)]:
    figura.add_trace(go.Scatter(x=z, y=curva, mode="lines", line=dict(color=cor), name=nome, showlegend=False), row=1, col=2)
figura.update_layout(height=340, margin=dict(l=10, r=10, t=50, b=10))
figura.show()
print("derivada maxima da sigmoide:", round(d_sig.max(), 3), "(em z=0)")

## 2. Sigmoide × ReLU no mesmo problema

Treinamos a mesma rede de duas luas trocando só a ativação oculta. A ReLU costuma convergir mais rápido.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler

X, y = make_moons(n_samples=400, noise=0.2, random_state=SEMENTE)
X = StandardScaler().fit_transform(X)
ent = torch.tensor(X, dtype=torch.float32)
alvo = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)
custo_fn = torch.nn.BCELoss()

figura = go.Figure()
for nome, ativacao, cor in [("sigmoide", torch.nn.Sigmoid(), AZUL), ("ReLU", torch.nn.ReLU(), VERMELHO)]:
    torch.manual_seed(SEMENTE)
    rede = torch.nn.Sequential(torch.nn.Linear(2,16), ativacao,
                               torch.nn.Linear(16,1), torch.nn.Sigmoid())
    oti = torch.optim.SGD(rede.parameters(), lr=0.1)
    perdas = []
    for epoca in range(200):
        perda = custo_fn(rede(ent), alvo)
        oti.zero_grad(); perda.backward(); oti.step()
        perdas.append(perda.item())
    figura.add_trace(go.Scatter(y=perdas, mode="lines", line=dict(color=cor), name=nome))
    print(nome.ljust(9), "perda final:", round(perdas[-1], 4))
figura.update_layout(title="Convergencia: sigmoide x ReLU", xaxis_title="epoca",
                     yaxis_title="perda", height=360, margin=dict(l=10, r=10, t=50, b=10))
figura.show()

## 3. O gradiente que desaparece, medido

Montamos uma rede **profunda** (6 camadas) e, após um único `backward`, medimos a magnitude do gradiente em cada camada. Com sigmoide, os gradientes das **primeiras** camadas são minúsculos; com ReLU, sobrevivem.

In [ ]:
for nome, ativacao in [("sigmoide", torch.nn.Sigmoid), ("ReLU", torch.nn.ReLU)]:
    torch.manual_seed(SEMENTE)
    camadas = []
    for c in range(6):
        camadas.append(torch.nn.Linear(16 if c else 2, 16))
        camadas.append(ativacao())
    camadas.append(torch.nn.Linear(16, 1)); camadas.append(torch.nn.Sigmoid())
    rede = torch.nn.Sequential(*camadas)
    perda = custo_fn(rede(ent), alvo)
    rede.zero_grad(); perda.backward()
    normas = []
    for camada in rede:
        if isinstance(camada, torch.nn.Linear):
            normas.append(camada.weight.grad.norm().item())
    print(nome.ljust(9), "norma do gradiente por camada (entrada -> saida):")
    print("   ", [round(v, 5) for v in normas])

## Exercício

No item 3, compare a norma do gradiente da **primeira** camada entre sigmoide e ReLU. O que esse número mostra sobre por que a ReLU virou padrão?

<details><summary>Ver resposta</summary>

Com **sigmoide**, a norma do gradiente na primeira camada é ordens de grandeza **menor** que nas últimas — o gradiente praticamente **desapareceu** ao atravessar as camadas, porque cada uma multiplicou por uma derivada < 1. Com **ReLU**, a derivada é 1 na região ativa, então o gradiente chega às primeiras camadas com magnitude útil e elas **conseguem aprender**. É exatamente por isso que a ReLU destravou o treino de redes profundas.

</details>